In [ ]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import shap, warnings
warnings.filterwarnings("ignore")

# ---------- 0. 路径 ----------
DATA_FILE   = "Depressing_scored.xlsx"
UNSCORED    = "images.xlsx"
OUT_DIR     = Path("D:\Desktop\output_Depressing"); 
OUT_DIR.mkdir(exist_ok=True)

PREDICT_XLSX = OUT_DIR / "images_predicted_Depressing.xlsx"
BSWARM_PNG   = OUT_DIR / "shap_beeswarm_Depressing.png"
BAR_PNG      = OUT_DIR / "shap_bar_Depressing.png"

# ---------- 1. 常量 ----------
TARGET = "Depressing"
FEATS  = ['Vegetation','Building','Riverway','Sky','Air Conditioner Condenser',
          'Clothes','Indicator','Road','Person','Car','Signboard',
          'Festival Elements','Motorcycle','Pole Group','Bicycle','Trash Can']
RF_PARAMS = dict(n_estimators=600, max_depth=11,
                 min_samples_leaf=2, max_features='sqrt',
                 random_state=42, n_jobs=-1)

# ---------- 2. 读取数据 ----------
df = pd.read_excel(DATA_FILE)
df[TARGET] = df[TARGET].round(5)

X = df[FEATS].copy()       
y = df[TARGET].values

# ---------- 3. 划分 & 随机森林 ----------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=42)

rf = RandomForestRegressor(**RF_PARAMS).fit(X_tr, y_tr)

# ---------- 4. 单调校准 ----------
iso = IsotonicRegression(
        y_min=y.min(), y_max=y.max(),
        increasing=True, out_of_bounds="clip"
     ).fit(rf.predict(X), y)

def metr(t,p): return r2_score(t,p), mean_squared_error(t,p,squared=False)
cal_tr = iso.transform(rf.predict(X_tr)); cal_te = iso.transform(rf.predict(X_te))
r2_tr, rmse_tr = metr(y_tr, cal_tr)
r2_te, rmse_te = metr(y_te, cal_te)
r2_all, rmse_all = metr(y, iso.transform(rf.predict(X)))


print(f"Train   R²={r2_tr :.3f} | RMSE={rmse_tr :.3f}")
print(f"Test    R²={r2_te :.3f} | RMSE={rmse_te :.3f}")
print(f"Overall R²={r2_all:.3f} | RMSE={rmse_all:.3f}")


# ---------- 5. SHAP ----------
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATS, show=False)
plt.title("SHAP Beeswarm — Depressing"); plt.tight_layout()
plt.savefig(BSWARM_PNG, bbox_inches="tight",dpi=300); plt.close()

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATS,
                  plot_type="bar", show=False)
plt.title("Feature Importance — Depressing")

plt.savefig(BAR_PNG, bbox_inches="tight",dpi=300); plt.close()

# ---------- 6. 预测未打分样本 ----------
df_new = pd.read_excel(UNSCORED)
X_new  = df_new[FEATS].copy()                
df_new[TARGET] = iso.transform(rf.predict(X_new)).round(5)
df_new.to_excel(PREDICT_XLSX, index=False, float_format="%.5f")

print("✔ 预测文件:", PREDICT_XLSX)
print("✔ SHAP 图:", BSWARM_PNG, BAR_PNG)


Train   R²=0.990 | RMSE=0.059
Test    R²=0.964 | RMSE=0.103
Overall R²=0.986 | RMSE=0.070
✔ 预测文件: D:\Desktop\output_Depressing2\images_predicted_Depressing.xlsx
✔ SHAP 图: D:\Desktop\output_Depressing2\shap_beeswarm_Depressing.png D:\Desktop\output_Depressing2\shap_bar_Depressing.png
